# Predictability of Non-Nearest-TSS Gold Standards

**Question:** 44.9% of gold-standard positives (GSP=1) are *not* the nearest
gene to the sentinel variant TSS. Does the feature matrix have enough signal
to distinguish them from negatives — or are they invisible to the model?

**Key finding upfront:** `distanceSentinelTssNeighbourhood` is a continuous
score (0–1, not binary). Score = 1.0 means nearest TSS; < 1.0 means not
nearest. The median score for non-nearest positives is **0.982** — most are
the *second-nearest* gene, not a distant outlier.

**Definitions:**
- **Nearest** (`score == 1.0`): 7,201 positives — 55.1% of all positives
- **Non-nearest** (`score < 1.0`): 5,859 positives — 44.9% of all positives
- **Negatives** (`GSP == 0`): 187,476 rows

In [ ]:
import pyspark.sql.functions as f
import pandas as pd
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )
    if Path(app_default_credentials).exists():
        return app_default_credentials
    raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)


session = GentropySession()

In [ ]:
col_tss = "distanceSentinelTssNeighbourhood"

fm = session.spark.read.parquet("2603_training_set_full_fm.parquet").cache()

nearest     = fm.filter((f.col("GSP") == 1) & (f.col(col_tss) == 1.0)).cache()
non_nearest = fm.filter((f.col("GSP") == 1) & (f.col(col_tss) <  1.0)).cache()
negatives   = fm.filter( f.col("GSP") == 0).cache()

n_near, n_far, n_neg = nearest.count(), non_nearest.count(), negatives.count()
n_pos = n_near + n_far

print(f"nearest     (score = 1.0) : {n_near:>7,}  ({n_near/n_pos*100:.1f}% of positives)")
print(f"non-nearest (score < 1.0) : {n_far:>7,}  ({n_far/n_pos*100:.1f}% of positives)")
print(f"  of which score = 0.0    : {non_nearest.filter(f.col(col_tss)==0.0).count():>7,}")
print(f"  of which 0 < score < 1  : {non_nearest.filter((f.col(col_tss)>0)&(f.col(col_tss)<1)).count():>7,}")
print(f"negatives                 : {n_neg:>7,}")

## 1. Nature of Non-Nearest Positives

In [ ]:
qs = non_nearest.approxQuantile(col_tss, [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0], 0.001)
print("distanceSentinelTssNeighbourhood — non-nearest positives:")
for q, v in zip(["min","p10","p25","p50","p75","p90","max"], qs):
    print(f"  {q}: {v:.4f}")
print()
print("Median = 0.982. 90% score above 0.86. Most are the second-nearest gene.")

## 2. QTL and Functional Signal Availability

In [ ]:
signal_features = {
    "eQTL_CLPP" : "eQtlColocClppMaximum",
    "eQTL_H4"   : "eQtlColocH4Maximum",
    "pQTL_CLPP" : "pQtlColocClppMaximum",
    "sQTL_CLPP" : "sQtlColocClppMaximum",
    "E2G"       : "e2gMean",
    "VEP > 0"   : "vepMaximum",
}

rows = []
for label, df, n in [("nearest (7,201)",     nearest,     n_near),
                     ("non-nearest (5,859)", non_nearest, n_far),
                     ("negatives (187,476)", negatives,   n_neg)]:
    row = {"group": label}
    for cat, col in signal_features.items():
        cnt = df.filter(f.coalesce(f.col(col), f.lit(0.0)) > 0).count()
        row[cat] = f"{cnt/n*100:.1f}%"
    rows.append(row)

pd.DataFrame(rows).set_index("group")

## 3. Signal-Dark Non-Nearest Positives

In [ ]:
no_signal = non_nearest.filter(
    (f.coalesce(f.col("eQtlColocClppMaximum"), f.lit(0.0)) == 0) &
    (f.coalesce(f.col("pQtlColocClppMaximum"), f.lit(0.0)) == 0) &
    (f.coalesce(f.col("sQtlColocClppMaximum"), f.lit(0.0)) == 0) &
    (f.coalesce(f.col("e2gMean"),              f.lit(0.0)) == 0)
)
dark, lit_up = no_signal.count(), n_far - no_signal.count()

print(f"No QTL or E2G signal : {dark:>5,} / {n_far:,}  ({dark/n_far*100:.1f}%)")
print(f"Has some signal      : {lit_up:>5,} / {n_far:,}  ({lit_up/n_far*100:.1f}%)")
print()
for cat, col in [("eQTL","eQtlColocClppMaximum"),("pQTL","pQtlColocClppMaximum"),
                 ("sQTL","sQtlColocClppMaximum"),("E2G", "e2gMean")]:
    cnt = non_nearest.filter(f.coalesce(f.col(col), f.lit(0.0)) > 0).count()
    print(f"  {cat:<6}: {cnt:>5,}  ({cnt/n_far*100:.1f}%)")

## 4. Separation from Negatives (Fold-Change)

In [ ]:
all_feats = ["eQtlColocClppMaximum","eQtlColocH4Maximum",
             "pQtlColocClppMaximum","sQtlColocClppMaximum",
             "e2gMean","vepMaximum"]

def group_means(df, label):
    row = df.agg(*[f.mean(f.coalesce(f.col(c),f.lit(0.0))).alias(c)
                   for c in all_feats]).collect()[0].asDict()
    row["group"] = label
    return row

means = pd.DataFrame([
    group_means(nearest,     "nearest"),
    group_means(non_nearest, "non-nearest"),
    group_means(negatives,   "negatives"),
]).set_index("group").round(5)

print("Mean feature values (nulls = 0):")
print(means.to_string())

eps = 1e-7
ratio = ((means.loc["non-nearest"] + eps) / (means.loc["negatives"] + eps)).round(1)
print()
print("Fold-change  non-nearest / negatives:")
print(ratio.sort_values(ascending=False).to_string())

## 5. eQTL Score Percentiles Across Groups

In [ ]:
qtl = "eQtlColocClppMaximum"
print(f"{'Group':<22}  p50     p75     p90     p95")
print("-" * 58)
for label, df in [("nearest (7,201)",     nearest),
                  ("non-nearest (5,859)", non_nearest),
                  ("negatives (187,476)", negatives)]:
    d = df.withColumn(qtl, f.coalesce(f.col(qtl), f.lit(0.0)))
    q = d.approxQuantile(qtl, [0.5, 0.75, 0.9, 0.95], 0.001)
    print(f"{label:<22}  {q[0]:.4f}  {q[1]:.4f}  {q[2]:.4f}  {q[3]:.4f}")

## Conclusions

### 1. The feature is continuous — most "non-nearest" are second-nearest
`distanceSentinelTssNeighbourhood` scores 0–1, with 1.0 = nearest. The
median for non-nearest positives is **0.982** and 90% score above 0.86.
Only 204 rows (3.5%) have score 0.0 (furthest gene). The group is
overwhelmingly second-nearest genes, not distant outliers.

### 2. 43.2% of non-nearest positives ARE distinguishable from negatives
QTL and E2G signal is strongly enriched in non-nearest positives
relative to negatives:

| Feature    | Non-nearest | Negatives | Fold-change (mean) |
|------------|-------------|-----------|-------------------|
| pQTL CLPP  | 10.0%       | 0.4%      | **23.6×**         |
| sQTL CLPP  | 19.3%       | 3.0%      | **24.9×**         |
| eQTL CLPP  | 25.4%       | 6.3%      | **9.7×**          |
| E2G        | 30.7%       | 7.7%      | **4.7×**          |

These examples explicitly teach the model to use functional evidence over
proximity — they are among the most informative examples in the set.

### 3. 56.8% are signal-dark — invisible beyond distance rank
3,325 / 5,859 non-nearest positives have zero value across all QTL and
E2G features. The model cannot distinguish them from negatives on those
features. They may be true effectors lacking QTL data in this release.

### 4. VEP does not help
VEP > 0 is true for ~100% of all groups; it has no discriminative power here.

### 5. E2G drops sharply for non-nearest
66.7% of nearest positives have E2G signal vs only **30.7%** for non-nearest.
Enhancer–gene links are naturally weaker for second-nearest/distal effectors.

### Recommendation
**Keep all non-nearest positives.** Removing the group would discard the
43.2% with strong QTL signal, biasing the model toward distance-only
predictions. The signal-dark 56.8% are not harmful — they are
uninformative rows that add minor noise.

Future work: run an ablation — train with and without the signal-dark
non-nearest positives — and measure precision at high L2G thresholds.